# Prepare Dataset

Converts the Amazon Reviews 2023 gzipped JSONL data (reviews + meta) into RecBole Atomic Files (.inter). 

In [2]:
import json
import gzip
from tqdm import tqdm
from pathlib import Path
from typing import Any
from collections.abc import Iterable
import pandas as pd

In [8]:
# --- Config ---
DATASET_NAME: str = "Beauty_and_Personal_Care"
CATEGORIES: list[str] = ["Beauty_and_Personal_Care"]
SAMPLE_SIZE: int | None = 100_000            # None = all rows
DATA_DIR: str = "../data"

# Only include reviews that satisfy all of the following criteria
START_DATE: str | None = "2020-01-01"
END_DATE: str | None = "2022-12-31"
MIN_RATING: int | None = None

# Data split ratios
TRAIN_RATIO: float = 0.8
VALID_RATIO: float = 0.1

# Cold vs. warm users
WARM_USER_MIN_REVIEWS: int = 5

# Sequential recommendation config
MAX_ITEM_LIST_LENGTH: int = 50

## Common utils

In [9]:
def stream_jsonl(path: str, fields: list[str] | None = None):
    if path.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as f:
            for _, line in enumerate(f):
                obj = json.loads(line)
                if fields is not None:
                    obj = {k: obj.get(k) for k in fields}
                yield obj
    else:
        with open(path, "r", encoding="utf-8") as f:
            for _, line in enumerate(f):
                obj = json.loads(line)
                if fields is not None:
                    obj = {k: obj.get(k) for k in fields}
                yield obj

def _date_to_ms(date_str: str | None) -> int | None:
    if date_str is None:
        return None
    return int(pd.Timestamp(date_str, tz="UTC").timestamp() * 1000)

def load_reviews(
    categories: list[str], sample_size: int | None = None, 
    start_date: str | None = None, end_date: str | None = None, 
    min_rating: int | None = None, gz_zip: bool = False
) -> list[dict[str, Any]]:
    start_ts = _date_to_ms(start_date)
    end_ts = _date_to_ms(end_date)
    print(f"Filtering reviews with criteria: start_date={start_date}, end_date={end_date}, min_rating={min_rating}")

    reviews: list[dict[str, Any]] = []
    for cat in categories:
        path = f"{DATA_DIR}/{cat}.jsonl" if not gz_zip else f"{DATA_DIR}/{cat}.jsonl.gz"
        print(f"Loading reviews: {path}")
        for obj in stream_jsonl(path, fields=[
            'user_id', 'parent_asin', 'rating', 'timestamp'
        ]):
            ts: Any = obj.get("timestamp")
            rating: Any = obj.get("rating")
            if start_ts is not None and (ts is not None and ts < start_ts):
                continue
            if end_ts is not None and (ts is not None and ts > end_ts):
                continue
            if min_rating is not None and (rating is not None and rating < min_rating):
                continue
            obj["category"] = cat
            reviews.append(obj)

            if sample_size is not None and len(reviews) >= sample_size:
                print(f"Reached sample size limit ({sample_size} reviews). Stopping.")
                break
    return reviews


## Load reviews

In [10]:
reviews = load_reviews(
    CATEGORIES, sample_size=SAMPLE_SIZE, start_date=START_DATE, end_date=END_DATE,
    min_rating=MIN_RATING, gz_zip=False
)
print(f"Loaded {len(reviews)} reviews")

Filtering reviews with criteria: start_date=2020-01-01, end_date=2022-12-31, min_rating=None
Loading reviews: ../data/Beauty_and_Personal_Care.jsonl
Reached sample size limit (100000 reviews). Stopping.
Loaded 100000 reviews


In [11]:
df_reviews = pd.DataFrame(reviews)
display(df_reviews.head())
df_reviews.info()

,user_id,parent_asin,rating,timestamp,category
0,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B00Z03RC80,1,1616743454733,Beauty_and_Personal_Care
1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B085PRT2MP,1,1614915977684,Beauty_and_Personal_Care
2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B08G81QQ9L,5,1612052493701,Beauty_and_Personal_Care
3,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07YYG76X1,1,1609700981786,Beauty_and_Personal_Care
4,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07X4FKLNK,3,1581313195358,Beauty_and_Personal_Care


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   user_id      100000 non-null  object
 1   parent_asin  100000 non-null  object
 2   rating       100000 non-null  int64 
 3   timestamp    100000 non-null  int64 
 4   category     100000 non-null  object
dtypes: int64(2), object(3)
memory usage: 3.8+ MB


## Split train/valid/test with cutoff timestamps

In [12]:
timestamps = sorted(df_reviews["timestamp"].values)
train_end_ts = timestamps[int(len(timestamps) * TRAIN_RATIO)]
valid_end_ts = timestamps[int(len(timestamps) * (TRAIN_RATIO + VALID_RATIO))]
print(f"Train end timestamp: {train_end_ts} ({pd.Timestamp(train_end_ts, unit='ms', tz='UTC')})")
print(f"Valid end timestamp: {valid_end_ts} ({pd.Timestamp(valid_end_ts, unit='ms', tz='UTC')})")

Train end timestamp: 1654641897128 (2022-06-07 22:44:57.128000+00:00)
Valid end timestamp: 1664376740827 (2022-09-28 14:52:20.827000+00:00)


In [13]:
df_reviews_train = df_reviews[df_reviews["timestamp"] <= train_end_ts]
df_reviews_valid = df_reviews[(df_reviews["timestamp"] > train_end_ts) & (df_reviews["timestamp"] <= valid_end_ts)]
df_reviews_test = df_reviews[df_reviews["timestamp"] > valid_end_ts]
print(f"Train reviews: {len(df_reviews_train)}")
print(f"Valid reviews: {len(df_reviews_valid)}")
print(f"Test reviews: {len(df_reviews_test)}")

Train reviews: 80001
Valid reviews: 10000
Test reviews: 9999


## Write .inter files (user-item interactions)

In [14]:
# As RecBole expects integer IDs, we need to create mappings from the original string IDs to integers.
user_ids: set[str] = set()
item_ids: set[str] = set()
for r in reviews:
    user_ids.add(r["user_id"])
    item_ids.add(r["parent_asin"])

user_map: dict[str, int] = {uid: i for i, uid in enumerate(sorted(user_ids))}
item_map: dict[str, int] = {pid: i for i, pid in enumerate(sorted(item_ids))}
print(f"Users: {len(user_map):,}  Items: {len(item_map):,}")

Users: 17,437  Items: 54,195


In [15]:
# write item map to file

item_map_path = Path(f"{DATA_DIR}/item_map.json")
with open(item_map_path, "w", encoding="utf-8") as f:
    json.dump(item_map, f)
print(f"Saved item map to {item_map_path}")

Saved item map to ../data/item_map.json


In [16]:
# write user map to file
user_map_path = Path(f"{DATA_DIR}/user_map.json")
with open(user_map_path, "w", encoding="utf-8") as f:
    json.dump(user_map, f)
print(f"Saved user map to {user_map_path}")

Saved user map to ../data/user_map.json


In [17]:
out_dir: Path = Path(DATA_DIR) / DATASET_NAME
prefix: Path = out_dir / DATASET_NAME

In [18]:
df_sorted: pd.DataFrame = pd.concat(
    [
        df_reviews_train.assign(split="train"),
        df_reviews_valid.assign(split="valid"),
        df_reviews_test.assign(split="test"),
    ],
    ignore_index=True,
).sort_values(
    ["user_id", "timestamp"],
    ascending=[True, True],
    kind="mergesort",
)

rows_by_split: dict[str, list[dict[str, str | int | float]]] = {
    "train": [],
    "valid": [],
    "test": [],
}

user_groups = df_sorted.groupby("user_id", sort=False)
for user_id, df_user in tqdm(
    user_groups,
    total=df_sorted["user_id"].nunique()
):
    history: list[int] = []

    for _, row in df_user.iterrows():
        uid: int | None = user_map.get(row["user_id"])
        iid: int | None = item_map.get(row["parent_asin"])
        if uid is None or iid is None:
            continue

        if history:
            split_name: str = str(row["split"])
            item_history: list[int] = history[-MAX_ITEM_LIST_LENGTH:]
            rows_by_split[split_name].append(
                {
                    "user_id": uid,
                    "item_id": iid,
                    "rating": float(row["rating"]),
                    "timestamp": float(row["timestamp"]),
                    "item_id_list": " ".join(str(item_id) for item_id in item_history),
                }
            )

        history.append(iid)

def write_inter_file(path: Path, rows: Iterable[dict[str, str | int | float]]) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    row_count: int = 0

    with path.open("w", encoding="utf-8") as f:
        f.write(
            "user_id:token\t"
            "item_id:token\t"
            "rating:float\t"
            "timestamp:float\t"
            "item_id_list:token_seq\n"
        )

        for row in rows:
            f.write(
                f"{row['user_id']}\t"
                f"{row['item_id']}\t"
                f"{row['rating']}\t"
                f"{row['timestamp']}\t"
                f"{row['item_id_list']}\n"
            )
            row_count += 1

    print(f"Wrote {path} ({row_count:,} rows)")

write_inter_file(prefix.with_suffix(".train.inter"), rows_by_split["train"])
write_inter_file(prefix.with_suffix(".valid.inter"), rows_by_split["valid"])
write_inter_file(prefix.with_suffix(".test.inter"), rows_by_split["test"])

100%|██████████| 17437/17437 [00:01<00:00, 9840.45it/s] 


Wrote ../data/Beauty_and_Personal_Care/Beauty_and_Personal_Care.train.inter (64,445 rows)
Wrote ../data/Beauty_and_Personal_Care/Beauty_and_Personal_Care.valid.inter (8,929 rows)
Wrote ../data/Beauty_and_Personal_Care/Beauty_and_Personal_Care.test.inter (9,189 rows)


## Analyze user interactions by split

In [19]:
train_counts = df_reviews_train.groupby("user_id", sort=False).size().rename("num_train")
valid_counts = df_reviews_valid.groupby("user_id", sort=False).size().rename("num_valid")
test_counts = df_reviews_test.groupby("user_id", sort=False).size().rename("num_test")

df_user_interactions = (
    pd.concat([train_counts, valid_counts, test_counts], axis=1)
    .fillna(0)
    .astype(int)
    .reset_index()
)

print("Sample of user interactions:")
display(df_user_interactions.sample(20))
print(f"\nTotal unique users: {len(df_user_interactions):,}")

Sample of user interactions:


,user_id,num_train,num_valid,num_test
16584,AHCAVQOYUJROXE2HTOLANN2MZTZQ,0,1,0
1618,AFQ4A4DZ6PYJYZJXKXCNEONVD24A,7,0,0
9121,AELTNW7ORL6XW3BREJ4RTKQHBVHA,13,1,0
16436,AEXRX4W7BMJ6HQ5UF7RD3ZVLICQQ,0,1,0
2594,AGMBFFN4HWYQYONQA7KI2CKYDJWA,1,0,1
4076,AFQTVWV7TN2F7LHFNJWHAGEOJ33Q,1,0,0
7330,AFAQ5BXMSW52757V6QYIQRKKPKGQ,1,0,0
13839,AHEBZHQAQR4PZTCFH2ZBPAXCVQEA,1,0,0
6262,AFWWR53BARLMR2MOE3TKAXHBN5SQ,2,2,0
14049,AGZJVRWVUC676ZFO4U3JERNLBB3Q,7,0,2



Total unique users: 17,437


## Define cold vs. warm users with train data

In [20]:
df_train_users = df_user_interactions[df_user_interactions["num_train"] > 0]
cold_user_ids: set[str] = set(df_train_users[df_train_users["num_train"] < WARM_USER_MIN_REVIEWS]["user_id"])
warm_user_ids: set[str] = set(df_train_users[df_train_users["num_train"] >= WARM_USER_MIN_REVIEWS]["user_id"])
print(f"Warm users (>= {WARM_USER_MIN_REVIEWS} reviews): {len(warm_user_ids)}")
print(f"Cold users (< {WARM_USER_MIN_REVIEWS} reviews): {len(cold_user_ids)}")

Warm users (>= 5 reviews): 2655
Cold users (< 5 reviews): 12901


## Write .user for user categories

In [21]:
def get_user_category(user_id: str) -> int:
    if user_id in warm_user_ids:
        return 0 # Warm user
    elif user_id in cold_user_ids:
        return 1 # Cold user
    else:
        return 2 # New user

def write_user_file(path: Path, rows: Iterable[dict[str, str | int | float]]) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    row_count: int = 0

    with path.open("w", encoding="utf-8") as f:
        f.write(
            "user_id:token\t"
            "category:token\n"
        )

        for row in rows:
            f.write(
                f"{row['user_id']}\t"
                f"{row['category']}\n"
            )
            row_count += 1

    print(f"Wrote {path} ({row_count:,} rows)")


write_user_file(prefix.with_suffix(".user"), [
    {
        "user_id": user_map[user_id],
        "category": get_user_category(user_id),
    }
    for user_id in df_user_interactions["user_id"]
])


Wrote ../data/Beauty_and_Personal_Care/Beauty_and_Personal_Care.user (17,437 rows)
